In [76]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
INPUT_PATH = PROJECT_ROOT / "data" / "input"
BRONZE_PATH = PROJECT_ROOT / "data" / "bronze"
SILVER_PATH = PROJECT_ROOT / "data" / "silver"
GOLD_PATH = PROJECT_ROOT / "data" / "gold"

# Test filepaths 
TEST_INPUT_PATH_B1 = PROJECT_ROOT / "data" / "test_input" / "batch_1"
TEST_INPUT_PATH_B2 = PROJECT_ROOT / "data" / "test_input" / "batch_2"

TEST_BRONZE_PATH = PROJECT_ROOT / "data" / "test_output" / "bronze"
TEST_SILVER_PATH = PROJECT_ROOT / "data" / "test_output" / "silver"
TEST_GOLD_PATH = PROJECT_ROOT / "data" / "test_output" / "gold"

print("Project root:", PROJECT_ROOT)
print("Input path:", INPUT_PATH)
print("Bronze path:", BRONZE_PATH)
print("Silver path:", SILVER_PATH)
print("Gold path:", GOLD_PATH)

Project root: c:\Users\Bruno\Desktop\git_repo\Olympic-dataplatform-challenge
Input path: c:\Users\Bruno\Desktop\git_repo\Olympic-dataplatform-challenge\data\input
Bronze path: c:\Users\Bruno\Desktop\git_repo\Olympic-dataplatform-challenge\data\bronze
Silver path: c:\Users\Bruno\Desktop\git_repo\Olympic-dataplatform-challenge\data\silver
Gold path: c:\Users\Bruno\Desktop\git_repo\Olympic-dataplatform-challenge\data\gold


In [57]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as f

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("OlympicsDevelopment")
    .getOrCreate()
)

spark

In [130]:
bronze_athlete_events = spark.read.parquet(f'{BRONZE_PATH}/athlete_events')
bronze_athlete_events.show(10, truncate=False)

bronze_athlete_events.filter((f.col('Team') == 'Singapore') & (f.col('NOC') != f.lit('SGP'))).show()
#bronze_athlete_events.count()
#bronze_athlete_events.filter(f.col('_corrupt_record').isNotNull()).show(10, truncate=False)
#bronze_athlete_events.filter(f.col('Event').isNull()).show(10, truncate=False)



+---+------------------------+---+---+------+------+--------------+---+-----------+----+------+-----------+-------------+----------------------------------+-----+---------------+--------------------------+------------------+
|ID |Name                    |Sex|Age|Height|Weight|Team          |NOC|Games      |Year|Season|City       |Sport        |Event                             |Medal|_corrupt_record|_ingested_at              |_source_filename  |
+---+------------------------+---+---+------+------+--------------+---+-----------+----+------+-----------+-------------+----------------------------------+-----+---------------+--------------------------+------------------+
|1  |A Dijiang               |M  |24 |180   |80    |China         |CHN|1992 Summer|1992|Summer|Barcelona  |Basketball   |Basketball Men's Basketball       |NA   |NULL           |2026-06-22 17:46:11.796249|athlete_events.csv|
|2  |A Lamusi                |M  |23 |170   |60    |China         |CHN|2012 Summer|2012|Summer|Londo

In [64]:
bronze_noc_regions = spark.read.parquet(f'{BRONZE_PATH}/noc_regions')
bronze_noc_regions.show(10, truncate=False)
#bronze_noc_regions.count()
#bronze_noc_regions.filter(f.col('_corrupt_record').isNotNull()).show(10, truncate=False)

+---+-----------+--------------------+---------------+-------------------------+----------------+
|NOC|region     |notes               |_corrupt_record|_ingested_at             |_source_filename|
+---+-----------+--------------------+---------------+-------------------------+----------------+
|AFG|Afghanistan|NULL                |NULL           |2026-06-22 15:51:03.24438|noc_regions.csv |
|AHO|Curacao    |Netherlands Antilles|NULL           |2026-06-22 15:51:03.24438|noc_regions.csv |
|ALB|Albania    |NULL                |NULL           |2026-06-22 15:51:03.24438|noc_regions.csv |
|ALG|Algeria    |NULL                |NULL           |2026-06-22 15:51:03.24438|noc_regions.csv |
|AND|Andorra    |NULL                |NULL           |2026-06-22 15:51:03.24438|noc_regions.csv |
|ANG|Angola     |NULL                |NULL           |2026-06-22 15:51:03.24438|noc_regions.csv |
|ANT|Antigua    |Antigua and Barbuda |NULL           |2026-06-22 15:51:03.24438|noc_regions.csv |
|ANZ|Australia  |Aus

In [125]:
silver_noc_regions = spark.read.parquet(f'{SILVER_PATH}/noc_regions')
silver_noc_regions.show(10, truncate=False)
silver_noc_regions.filter(f.col('noc_code') == 'SIN').show()
silver_noc_regions.filter((f.col('region') == 'Singapore') & f.col('NOC') != f.lit('SIN')).show()

+--------+-----------+--------------------+--------------------------+----------------+----------------------------------------------------------------+
|noc_code|region     |notes               |_ingested_at              |_source_filename|record_hash                                                     |
+--------+-----------+--------------------+--------------------------+----------------+----------------------------------------------------------------+
|AFG     |Afghanistan|NULL                |2026-06-22 17:46:14.164719|noc_regions.csv |5dbddf911f9e565299428e948a92ea5d1943c4099db883cc4491cb32cc757de8|
|AHO     |Curacao    |Netherlands Antilles|2026-06-22 17:46:14.164719|noc_regions.csv |3fe62e28937ed8f2f36975d0fb96e221b7658227a9adfbbaf9fd2182d083be3c|
|ALB     |Albania    |NULL                |2026-06-22 17:46:14.164719|noc_regions.csv |1af00b874ff3183021ca5c541e6f3e8d1bbb2b412bdb87f44cb07ae930dad646|
|ALG     |Algeria    |NULL                |2026-06-22 17:46:14.164719|noc_regions.

In [127]:
silver_athlete_events = spark.read.parquet(f'{SILVER_PATH}/athlete_events')
silver_athlete_events.show(10, truncate=False)
silver_athlete_events.filter(f.col('noc_code') == 'SGP').show()
silver_athlete_events.filter(f.col('team') == 'Singapore').show()

+----------+--------------------------------+---+---+---------+---------+-------------+--------+-----------+----+------+--------------+----------+----------------------------------------+------+--------------------------+------------------+----------------------------------------------------------------+
|athlete_id|athlete_name                    |sex|age|height_cm|weight_kg|team         |noc_code|games_name |year|season|host_city     |sport     |event_name                              |medal |_ingested_at              |_source_filename  |source_record_hash                                              |
+----------+--------------------------------+---+---+---------+---------+-------------+--------+-----------+----+------+--------------+----------+----------------------------------------+------+--------------------------+------------------+----------------------------------------------------------------+
|16        |Juhamatti Tapio Aaltonen        |M  |28 |184.0    |85.0     |Finland  

In [70]:
dim_athlete = spark.read.parquet(f'{GOLD_PATH}/dim_athlete')
dim_athlete.show(10, truncate=False)

+--------------------+----------+------------------------------+---+
|athlete_key         |athlete_id|athlete_name                  |sex|
+--------------------+----------+------------------------------+---+
|-7001672635703045582|1         |A Dijiang                     |M  |
|8792594768435660362 |6         |Per Knut Aaland               |M  |
|-1938844103824896804|9         |Antti Sami Aalto              |M  |
|1833446804176943981 |10        |Einar Ferdinand "Einari" Aalto|M  |
|5217482232275368447 |11        |Jorma Ilmari Aalto            |M  |
|-7449598549972235930|14        |Pirjo Hannele Aalto (Mattila-)|F  |
|5486548753499319695 |16        |Juhamatti Tapio Aaltonen      |M  |
|5840989680794618359 |17        |Paavo Johannes Aaltonen       |M  |
|715422782555130402  |18        |Timo Antero Aaltonen          |M  |
|-1931231880553328050|22        |Andreea Aanei                 |F  |
+--------------------+----------+------------------------------+---+
only showing top 10 rows


In [71]:
dim_games = spark.read.parquet(f'{GOLD_PATH}/dim_games')
dim_games.show(10, truncate=False)

+--------------------+-----------+----+------+---------+
|games_key           |games_name |year|season|host_city|
+--------------------+-----------+----+------+---------+
|8899999260635012312 |1896 Summer|1896|Summer|Athina   |
|267977890873369262  |1900 Summer|1900|Summer|Paris    |
|2376459188086408989 |1904 Summer|1904|Summer|St. Louis|
|-977336073213603169 |1906 Summer|1906|Summer|Athina   |
|-8375417704997512265|1908 Summer|1908|Summer|London   |
|-7715366089314896380|1912 Summer|1912|Summer|Stockholm|
|-2785993201817665857|1920 Summer|1920|Summer|Antwerpen|
|7472552201212159650 |1924 Summer|1924|Summer|Paris    |
|-2009256158056512548|1924 Winter|1924|Winter|Chamonix |
|7547280808084468973 |1928 Summer|1928|Summer|Amsterdam|
+--------------------+-----------+----+------+---------+
only showing top 10 rows


In [72]:
dim_event = spark.read.parquet(f'{GOLD_PATH}/dim_event')
dim_event.show(10, truncate=False)

+--------------------+-------------+--------------------------------------+
|event_key           |sport        |event_name                            |
+--------------------+-------------+--------------------------------------+
|8986519178593494313 |Basketball   |Basketball Men's Basketball           |
|9012481418576441575 |Judo         |Judo Men's Extra-Lightweight          |
|-6138819449700194079|Tug-Of-War   |Tug-Of-War Men's Tug-Of-War           |
|-8262739988155263573|Speed Skating|Speed Skating Women's 500 metres      |
|8521857191793979263 |Ice Hockey   |Ice Hockey Men's Ice Hockey           |
|-881679395300882502 |Swimming     |Swimming Men's 400 metres Freestyle   |
|1538835383841364551 |Swimming     |Swimming Men's 200 metres Breaststroke|
|1544701580199534423 |Gymnastics   |Gymnastics Men's Individual All-Around|
|2696320382474620872 |Gymnastics   |Gymnastics Men's Parallel Bars        |
|-7944213220176682002|Gymnastics   |Gymnastics Men's Rings                |
+-----------

In [73]:
dim_noc = spark.read.parquet(f'{GOLD_PATH}/dim_noc')
dim_noc.show(10, truncate=False)

+--------------------+--------+--------------+-----+----------------------------------------------------------------+----------+----------+----------+
|noc_key             |noc_code|region        |notes|record_hash                                                     |valid_from|valid_to  |is_current|
+--------------------+--------+--------------+-----+----------------------------------------------------------------+----------+----------+----------+
|8341203863923922563 |ALG     |Algeria       |NULL |30e230229746a0c64818c9bc8f2c86e31bbb500f0a8db33c4b02adf9cded18f8|2026-06-21|9999-12-31|true      |
|7994735998091465814 |AZE     |Azerbaijan    |NULL |efa9150afdeaa69ffcdefc990cb8db6e67ac2aed628d8c696da4e36ed649f39b|2026-06-21|9999-12-31|true      |
|-4074788188975601739|BAN     |Bangladesh    |NULL |2f1d47eb679bb05e6d8fe039af1563549588cd44b40be6b1013ea8ca2268845f|2026-06-21|9999-12-31|true      |
|2940787267598443517 |BEN     |Benin         |NULL |69d0a7b84cd1d5d0928e636b0a7204897af58fe55b

In [74]:
fact_part = spark.read.parquet(f'{GOLD_PATH}/fact_participation')
fact_part.show(10, truncate=False)

+----------------------------------------------------------------+-------------------+--------------------+--------------------+--------------------+-------------+---+---------+---------+------+-------------------+-----------+----------+------------+------------+
|participation_key                                               |athlete_key        |games_key           |event_key           |noc_key             |team         |age|height_cm|weight_kg|medal |participation_count|medal_count|gold_count|silver_count|bronze_count|
+----------------------------------------------------------------+-------------------+--------------------+--------------------+--------------------+-------------+---+---------+---------+------+-------------------+-----------+----------+------------+------------+
|d12cf7eeb733f21a4ea9aba09f15fe5cbda8976d93262b9e5641b34475330990|2692252814664841420|-50696337927509860  |-5426625005134060151|-1841691976118184881|Norway       |22 |176.0    |85.0     |Silver|1             

# FIRST RUN

In [113]:

dim_noc_b1 = spark.read.parquet(f'{TEST_GOLD_PATH}/dim_noc') 

dim_b1 = dim_noc_b1

dim_b1.show(truncate=False)

+--------------------+--------+-------------+-----+----------------------------------------------------------------+----------+----------+----------+
|noc_key             |noc_code|region       |notes|record_hash                                                     |valid_from|valid_to  |is_current|
+--------------------+--------+-------------+-----+----------------------------------------------------------------+----------+----------+----------+
|4764428838837529480 |USA     |United States|NULL |49dca65f362fee401292ed7ada96f96295eab1e589c52e4e66bf4aedda715fdd|2026-06-20|9999-12-31|true      |
|-8330370774612027490|POR     |Portugal     |NULL |4c7de2c3da6dc0ae8f9d13b107718a913859359bd30166d077c867577953865d|2026-06-20|9999-12-31|true      |
+--------------------+--------+-------------+-----+----------------------------------------------------------------+----------+----------+----------+



In [114]:
fact_b1 = spark.read.parquet(f'{TEST_GOLD_PATH}/fact_participation')

fact_b1.show(truncate=False)
fact_b1.count()

+----------------------------------------------------------------+--------------------+-------------------+-------------------+--------------------+-------------+---+---------+---------+------+-------------------+-----------+----------+------------+------------+
|participation_key                                               |athlete_key         |games_key          |event_key          |noc_key             |team         |age|height_cm|weight_kg|medal |participation_count|medal_count|gold_count|silver_count|bronze_count|
+----------------------------------------------------------------+--------------------+-------------------+-------------------+--------------------+-------------+---+---------+---------+------+-------------------+-----------+----------+------------+------------+
|a90f9704f6dd9f27e6c396cb4d23b63630b5022596cc0f2f32027ef01cc320fc|-3341702809300393011|8238027849333505837|415971125356406525 |4764428838837529480 |United States|25 |185.0    |80.0     |Gold  |1                 

3

In [116]:
dim_noc_b2 = spark.read.parquet(f'{TEST_GOLD_PATH}/dim_noc') 
dim_noc_b2.show(truncate=False)

+--------+--------------------+-------------------+-----+----------------------------------------------------------------+----------+----------+----------+
|noc_code|noc_key             |region             |notes|record_hash                                                     |valid_from|valid_to  |is_current|
+--------+--------------------+-------------------+-----+----------------------------------------------------------------+----------+----------+----------+
|POR     |4584569395892189058 |Portuguese Republic|NULL |8747632b73d90613ccf4dd07277169573fa71bf6e25b5fdae99773b4b80e26e5|2026-06-22|9999-12-31|true      |
|USA     |4764428838837529480 |United States      |NULL |49dca65f362fee401292ed7ada96f96295eab1e589c52e4e66bf4aedda715fdd|2026-06-20|9999-12-31|true      |
|POR     |-8330370774612027490|Portugal           |NULL |4c7de2c3da6dc0ae8f9d13b107718a913859359bd30166d077c867577953865d|2026-06-20|2026-06-21|false     |
|DEN     |-1831944265744799804|Denmark            |NULL |c527097

In [115]:
fact_b2 = spark.read.parquet(f'{TEST_GOLD_PATH}/fact_participation')

fact_b2.show(truncate=False)
fact_b2.count()

+----------------------------------------------------------------+--------------------+-------------------+-------------------+--------------------+--------------+---+---------+---------+------+-------------------+-----------+----------+------------+------------+
|participation_key                                               |athlete_key         |games_key          |event_key          |noc_key             |team          |age|height_cm|weight_kg|medal |participation_count|medal_count|gold_count|silver_count|bronze_count|
+----------------------------------------------------------------+--------------------+-------------------+-------------------+--------------------+--------------+---+---------+---------+------+-------------------+-----------+----------+------------+------------+
|a90f9704f6dd9f27e6c396cb4d23b63630b5022596cc0f2f32027ef01cc320fc|-3341702809300393011|8238027849333505837|415971125356406525 |4764428838837529480 |United States |25 |185.0    |80.0     |Gold  |1             

4

In [96]:
dim_athlete_b1 = spark.read.parquet(f'{TEST_GOLD_PATH}/dim_athlete') 

dim_athlete_b1.show(truncate=False)

+--------------------+----------+------------+---+
|athlete_key         |athlete_id|athlete_name|sex|
+--------------------+----------+------------+---+
|-7001672635703045582|1         |Ana Silva   |F  |
|-3341702809300393011|2         |John Smith  |M  |
|3188756510806108107 |3         |Maria Costa |F  |
+--------------------+----------+------------+---+



In [97]:
dim_event_b1 = spark.read.parquet(f'{TEST_GOLD_PATH}/dim_event') 

dim_event_b1.show(truncate=False)

+-------------------+---------+-----------------------------------+
|event_key          |sport    |event_name                         |
+-------------------+---------+-----------------------------------+
|7954763747419682928|Judo     |Judo Women's Lightweight           |
|8714404582084013114|Athletics|Athletics Women's 100 metres       |
|415971125356406525 |Swimming |Swimming Men's 100 metres Freestyle|
+-------------------+---------+-----------------------------------+



In [98]:
dim_games_b1 = spark.read.parquet(f'{TEST_GOLD_PATH}/dim_games') 

dim_games_b1.show(truncate=False)

+-------------------+-----------+----+------+--------------+
|games_key          |games_name |year|season|host_city     |
+-------------------+-----------+----+------+--------------+
|8238027849333505837|2016 Summer|2016|Summer|Rio de Janeiro|
|8810379344728962529|2020 Summer|2020|Summer|Tokyo         |
+-------------------+-----------+----+------+--------------+



In [99]:
fact_b1 = spark.read.parquet(f'{TEST_GOLD_PATH}/fact_participation')

fact_b1.show(truncate=False)
fact_b1.count()

+----------------------------------------------------------------+--------------------+-------------------+-------------------+--------------------+-------------+---+---------+---------+------+-------------------+-----------+----------+------------+------------+
|participation_key                                               |athlete_key         |games_key          |event_key          |noc_key             |team         |age|height_cm|weight_kg|medal |participation_count|medal_count|gold_count|silver_count|bronze_count|
+----------------------------------------------------------------+--------------------+-------------------+-------------------+--------------------+-------------+---+---------+---------+------+-------------------+-----------+----------+------------+------------+
|a90f9704f6dd9f27e6c396cb4d23b63630b5022596cc0f2f32027ef01cc320fc|-3341702809300393011|8238027849333505837|415971125356406525 |4764428838837529480 |United States|25 |185.0    |80.0     |Gold  |1                 

3

# Second run

In [100]:
dim_noc_b2 = spark.read.parquet(f'{TEST_GOLD_PATH}/dim_noc') 

dim_noc_b2.orderBy(
    "noc_code",
    "valid_from",
).show(truncate=False)

+--------+--------------------+-------------------+-----+----------------------------------------------------------------+----------+----------+----------+
|noc_code|noc_key             |region             |notes|record_hash                                                     |valid_from|valid_to  |is_current|
+--------+--------------------+-------------------+-----+----------------------------------------------------------------+----------+----------+----------+
|DEN     |-4268295289538763759|Denmark            |NULL |c527097577600a6726de99bd6f555fbfaf214da9b7d127d7dd4a807927422b98|2026-06-21|9999-12-31|true      |
|POR     |-8330370774612027490|Portugal           |NULL |4c7de2c3da6dc0ae8f9d13b107718a913859359bd30166d077c867577953865d|2026-06-20|2026-06-20|false     |
|POR     |-1449890261865743667|Portuguese Republic|NULL |8747632b73d90613ccf4dd07277169573fa71bf6e25b5fdae99773b4b80e26e5|2026-06-21|9999-12-31|true      |
|USA     |4764428838837529480 |United States      |NULL |49dca65

In [101]:
dim_athlete_b2 = spark.read.parquet(f'{TEST_GOLD_PATH}/dim_athlete') 

dim_athlete_b2.show(truncate=False)

+--------------------+----------+-----------------------------------------------------------------+---+
|athlete_key         |athlete_id|athlete_name                                                     |sex|
+--------------------+----------+-----------------------------------------------------------------+---+
|-7001672635703045582|1         |Ana Silva                                                        |F  |
|-3341702809300393011|2         |John Smith                                                       |M  |
|3188756510806108107 |3         |Maria Costa                                                      |F  |
|404280023041566627  |4         |Myra Abigail "Abbie" Pratt (Pankhurst-, Wright-, -Karageorgevich)|F  |
+--------------------+----------+-----------------------------------------------------------------+---+



In [102]:
dim_event_b2 = spark.read.parquet(f'{TEST_GOLD_PATH}/dim_event') 

dim_event_b2.show(truncate=False)

+-------------------+---------+-----------------------------------+
|event_key          |sport    |event_name                         |
+-------------------+---------+-----------------------------------+
|7954763747419682928|Judo     |Judo Women's Lightweight           |
|8714404582084013114|Athletics|Athletics Women's 100 metres       |
|415971125356406525 |Swimming |Swimming Men's 100 metres Freestyle|
|2947087390348342364|Sailing  |Sailing Mixed 2-3 Ton              |
+-------------------+---------+-----------------------------------+



In [103]:
dim_games_b2 = spark.read.parquet(f'{TEST_GOLD_PATH}/dim_games') 

dim_games_b2.show(truncate=False)

+-------------------+-----------+----+------+--------------+
|games_key          |games_name |year|season|host_city     |
+-------------------+-----------+----+------+--------------+
|267977890873369262 |1900 Summer|1900|Summer|Paris         |
|8238027849333505837|2016 Summer|2016|Summer|Rio de Janeiro|
|8810379344728962529|2020 Summer|2020|Summer|Tokyo         |
+-------------------+-----------+----+------+--------------+



In [104]:
fact_b2 = spark.read.parquet(f'{TEST_GOLD_PATH}/fact_participation')

fact_b2.show(truncate=False)
fact_b2.count()

+----------------------------------------------------------------+--------------------+-------------------+-------------------+--------------------+--------------+---+---------+---------+------+-------------------+-----------+----------+------------+------------+
|participation_key                                               |athlete_key         |games_key          |event_key          |noc_key             |team          |age|height_cm|weight_kg|medal |participation_count|medal_count|gold_count|silver_count|bronze_count|
+----------------------------------------------------------------+--------------------+-------------------+-------------------+--------------------+--------------+---+---------+---------+------+-------------------+-----------+----------+------------+------------+
|a90f9704f6dd9f27e6c396cb4d23b63630b5022596cc0f2f32027ef01cc320fc|-3341702809300393011|8238027849333505837|415971125356406525 |4764428838837529480 |United States |25 |185.0    |80.0     |Gold  |1             

4

In [107]:
dim_noc_b2.orderBy(
    "noc_code",
    "valid_from",
).show(truncate=False)


+--------+--------------------+-------------------+-----+----------------------------------------------------------------+----------+----------+----------+
|noc_code|noc_key             |region             |notes|record_hash                                                     |valid_from|valid_to  |is_current|
+--------+--------------------+-------------------+-----+----------------------------------------------------------------+----------+----------+----------+
|DEN     |-4268295289538763759|Denmark            |NULL |c527097577600a6726de99bd6f555fbfaf214da9b7d127d7dd4a807927422b98|2026-06-21|9999-12-31|true      |
|POR     |-8330370774612027490|Portugal           |NULL |4c7de2c3da6dc0ae8f9d13b107718a913859359bd30166d077c867577953865d|2026-06-20|2026-06-20|false     |
|POR     |-1449890261865743667|Portuguese Republic|NULL |8747632b73d90613ccf4dd07277169573fa71bf6e25b5fdae99773b4b80e26e5|2026-06-21|9999-12-31|true      |
|USA     |4764428838837529480 |United States      |NULL |49dca65

In [131]:
spark.stop()